In [1]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.optim import Optimizer, Adam, AdamW

/root/KartikGoyal/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
GPU_ID = "cuda:1"
model_id = "meta-llama/Llama-3.2-3B" # or "Qwen/Qwen3.5-9B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, device_map = {"":GPU_ID}, dtype = torch.float16)


RuntimeError: No CUDA GPUs are available

In [3]:
param_size = 0
for param in model.parameters():
    param_size += param.nelement() * param.element_size()

buffer_size = 0
for buffer in model.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

size_all_mb = (param_size + buffer_size) / 1024**2
print('Model size: {:.3f}MB'.format(size_all_mb))


Model size: 6127.834MB


In [ ]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072,), eps=1e-05)
    (

In [ ]:
dummy_text = [
    "Hello this is a test sentence for the llama model.",
    "Transformers predict the next token in sequence.",
    "This is dummy training data for causal language modeling.",
    "Testing forward pass with synthetic inputs."
]
tokens = tokenizer(dummy_text ,
                   padding="max_length",
                    truncation=True,
                    max_length=32,
                    return_tensors="pt",)
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
model.eval()
device = torch.device(GPU_ID)
model.to(device)
tokens.to(device)


{'input_ids': tensor([[128000,   9906,    420,    374,    264,   1296,  11914,    369,    279,
          94776,   1646,     13, 128001, 128001, 128001, 128001, 128001, 128001,
         128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001,
         128001, 128001, 128001, 128001, 128001],
        [128000,   9140,    388,   7168,    279,   1828,   4037,    304,   8668,
             13, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001,
         128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001,
         128001, 128001, 128001, 128001, 128001],
        [128000,   2028,    374,  17741,   4967,    828,    369,  59557,   4221,
          34579,     13, 128001, 128001, 128001, 128001, 128001, 128001, 128001,
         128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001,
         128001, 128001, 128001, 128001, 128001],
        [128000,  16856,   4741,   1522,    449,  28367,  11374,     13, 128001,
         128001, 128001, 1

In [ ]:
def _shift_labels(input_ids, attention_mask):
    labels = input_ids.clone()
    labels[:,:-1] = input_ids[:,1:]
    labels[:,-1] = -100

    labels = labels.masked_fill(attention_mask == 0, -100)

    return labels

In [ ]:
input_ids = tokens['input_ids']
attention_mask = tokens['attention_mask']

labels = _shift_labels(input_ids, attention_mask)


In [ ]:
attention_mask[0]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:1')

In [ ]:
print("Input Shape: ", input_ids.shape)
print("Mask Shape: " , attention_mask.shape)
print("Labels Shape: ", labels.shape)

Input Shape:  torch.Size([4, 32])
Mask Shape:  torch.Size([4, 32])
Labels Shape:  torch.Size([4, 32])


In [ ]:
input_ids[0]

tensor([128000,   9906,    420,    374,    264,   1296,  11914,    369,    279,
         94776,   1646,     13, 128001, 128001, 128001, 128001, 128001, 128001,
        128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001, 128001,
        128001, 128001, 128001, 128001, 128001], device='cuda:1')

In [ ]:
labels[0]

tensor([  9906,    420,    374,    264,   1296,  11914,    369,    279,  94776,
          1646,     13, 128001,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100], device='cuda:1')

In [ ]:
model.train()

output = model(input_ids = input_ids, attention_mask = attention_mask, labels = labels)
loss = output.loss
print(loss)
loss.backward()

tensor(9.3026, device='cuda:1', grad_fn=<NllLossBackward0>)


In [ ]:
from torch.optim import SGD

In [ ]:
optimizer = SGD(model.parameters(),lr = 1e-4)


In [ ]:
optimizer.step()